# SAM3-I：Colab 单图指令分割

本 notebook 使用随附的 `sam3i-colab-source.zip`，不依赖尚未推送的本地提交。作者的模型文件保持原样；独立适配层使用原构建器、输入处理与前向，仅将 checkpoint 读取改为 mmap、掩码放大改到 CPU。

**本地已验证的范围：CPU 接口、FP32 合成后处理、真实权重名称/形状。用户于 2026-11-03 确认 Colab 单图推理成功；具体 GPU、精度和运行日志尚未记录。**

先在 Colab 菜单“运行时 → 更改运行时类型”选择 GPU。模型 FP32 参数约 6.78 GiB，激活另需显存；优先选择显存与主机内存较大的可用运行时。T4/FP16 需要实测。GPU 类型与可用资源不保证，见 [Colab FAQ](https://research.google.com/colaboratory/faq.html)。

准备：本 notebook、源码 ZIP、作者 Stage 3 checkpoint（约 9.2 GiB，放自己的 Google Drive）和一张图片。无需下载评估数据集或 SAM3 基础权重。

## 1. 上传本地部署源码包

只上传 `sam3i-colab-source.zip`，此处不要上传 checkpoint。核对打印出的 SHA256 与本地构建输出一致。

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from google.colab import files

uploaded_source = files.upload()
if len(uploaded_source) != 1:
    raise ValueError("请选择且只选择 sam3i-colab-source.zip")
source_name, source_bytes = next(iter(uploaded_source.items()))
if not source_name.endswith(".zip"):
    raise ValueError("需要源码 ZIP")
source_sha256 = hashlib.sha256(source_bytes).hexdigest()
print("Source ZIP SHA256:", source_sha256)
source_zip = Path("/content") / f"sam3i-source-{source_sha256[:12]}.zip"
source_zip.write_bytes(source_bytes)
del source_bytes, uploaded_source
deployment_root = Path("/content") / f"sam3i-deploy-{source_sha256[:12]}"
deployment_root.mkdir(exist_ok=True)
with zipfile.ZipFile(source_zip) as archive:
    for member in archive.infolist():
        target = (deployment_root / member.filename).resolve()
        if not target.is_relative_to(deployment_root.resolve()):
            raise ValueError("源码 ZIP 路径越界")
    archive.extractall(deployment_root)
REPO = deployment_root / "SAM3-I"
manifest = json.loads((REPO / "SOURCE_MANIFEST.json").read_text())
for relative, digest in manifest["sha256"].items():
    path = (REPO / relative).resolve()
    if not path.is_relative_to(REPO.resolve()):
        raise ValueError("清单路径越界")
    if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
        raise ValueError(f"源码校验失败：{relative}")
print("Verified files:", len(manifest["sha256"]))
print("Base local Git HEAD:", manifest["git_head"])
print("Actual snapshot is identified by per-file hashes, including local additions.")


## 2. 创建项目自己的 Python 3.12 / uv 环境

Colab 原生内核只负责 Drive、上传和展示；项目通过子进程运行。首次需要下载 PyTorch 等依赖。使用项目锁文件，不启用视频或 notebooks 依赖组。

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
RUN_ENV = dict(os.environ)
RUN_ENV.update({
    "UV_CACHE_DIR": "/content/.cache/sam3i-uv",
    "HF_HOME": "/content/.cache/sam3i-hf",
    "MPLCONFIGDIR": "/content/.cache/sam3i-matplotlib",
    "PYTHONUNBUFFERED": "1",
    "CUDA_VISIBLE_DEVICES": "0",
})
subprocess.run(UV + ["sync", "--locked", "--no-dev", "--project", str(REPO)],
               env=RUN_ENV, check=True)
RUN = UV + ["run", "--locked", "--no-dev", "--project", str(REPO), "python"]
subprocess.run(RUN + ["-c",
    "import torch; print('torch:', torch.__version__); "
    "assert torch.cuda.is_available(), '请启用 GPU 运行时'; "
    "print('GPU:', torch.cuda.get_device_name(0)); "
    "print('free/total GiB:', [round(x/2**30, 2) for x in torch.cuda.mem_get_info()]); "
    "print('native BF16:', torch.cuda.is_bf16_supported(including_emulation=False))"],
    env=RUN_ENV, check=True)


## 3. 挂载自己的 Drive，并复制 Stage 3 权重到本地磁盘

修改 `DRIVE_CHECKPOINT`。这里保持作者 checkpoint 的内容和文件名；使用 `inst_stage="3"` 避免阶段迁移。复制到 `/content` 后再 mmap，不直接 mmap Drive 挂载文件。只使用你信任的作者权重，读取行为保留作者的 `weights_only=False`。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CHECKPOINT = "/content/drive/MyDrive/SAM3-I/checkpoint.pt" # @param {type:"string"}
checkpoint_source = Path(DRIVE_CHECKPOINT)
if not checkpoint_source.is_file():
    raise FileNotFoundError(f"请修改权重路径：{checkpoint_source}")
CHECKPOINT = Path("/content/sam3i-checkpoints/checkpoint.pt")
CHECKPOINT.parent.mkdir(exist_ok=True)
if checkpoint_source.resolve() != CHECKPOINT.resolve():
    required = checkpoint_source.stat().st_size + 1024**3
    if shutil.disk_usage(CHECKPOINT.parent).free < required:
        raise RuntimeError("本地磁盘空间不足，需为 checkpoint 副本预留至少文件大小 + 1 GiB")
    shutil.copy2(checkpoint_source, CHECKPOINT)
print("Local checkpoint GiB:", round(CHECKPOINT.stat().st_size / 2**30, 3))


## 4. 在 CPU/meta 上核对权重结构

该子进程显式隐藏 GPU，不读取权重数值，不执行模型前向。名称或形状不匹配时停止。Colab 原生内核仍可正常使用 GPU。

In [ ]:
audit_env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES="")
AUDIT_PATH = Path("/content/sam3i-checkpoint-audit.json")
subprocess.run(RUN + [
    str(REPO / "scripts/audit_single_image_checkpoint.py"),
    "--checkpoint", str(CHECKPOINT), "--output", str(AUDIT_PATH),
], env=audit_env, check=True, stdout=subprocess.DEVNULL)
audit = json.loads(AUDIT_PATH.read_text())
print("Audit OK:", audit["ok"])
print("Forward keys:", audit["matched_state_keys"], "/", audit["expected_state_keys"])
print("Loading topology:", audit["real_checkpoint_loading_topology"]["matched_state_keys"],
      "/", audit["real_checkpoint_loading_topology"]["state_keys"])
print("CUDA initialized:", audit["cuda_initialized_after"])


## 5. 上传一张图片

In [ ]:
from PIL import Image as PILImage
from IPython.display import display

uploaded_images = files.upload()
if len(uploaded_images) != 1:
    raise ValueError("请选择且只选择一张图片")
image_name, image_bytes = next(iter(uploaded_images.items()))
IMAGE_PATH = Path("/content/sam3i-input") / Path(image_name).name
IMAGE_PATH.parent.mkdir(exist_ok=True)
IMAGE_PATH.write_bytes(image_bytes)
del image_bytes, uploaded_images
with PILImage.open(IMAGE_PATH) as uploaded_image:
    print("Stored image size (W,H):", uploaded_image.size)
    display(uploaded_image.copy().resize((min(800, uploaded_image.width),
            max(1, round(uploaded_image.height * min(800, uploaded_image.width) / uploaded_image.width)))))


## 6. 单图推理

提示类型由你显式选择，不根据句子长度自动判断：

| category | 路径 | 提示示例 |
|---|---|---|
| concept | text_type=1 | shoe |
| simple | text_type=2 | the red shoe in the top row |
| complex | text_type=4 | the red item in the top row worn on feet while running |

默认用英文测试。自动精度在原生 BF16 可用时选 BF16，否则 FP16；FP16 数值与效果尚待 GPU 实测。可手动设 `fp32` 排查数值问题，但会增加显存压力。

保留分数高于阈值的全部实例，不自动取 top-1。原图按作者流程缩放至 1008×1008 计算，掩码在 CPU 逐个恢复原图尺寸。每次单元格调用会重新加载模型，子进程退出后释放模型显存。

In [ ]:
import uuid
PROMPT = "the red shoe in the top row" # @param {type:"string"}
CATEGORY = "simple" # @param ["concept", "simple", "complex"]
THRESHOLD = 0.5 # @param {type:"number"}
PRECISION = "auto" # @param ["auto", "bf16", "fp16", "fp32"]
OUTPUT = Path("/content/sam3i-results") / uuid.uuid4().hex[:12]
subprocess.run(RUN + [
    str(REPO / "scripts/colab_single_image.py"),
    "--checkpoint", str(CHECKPOINT), "--image", str(IMAGE_PATH),
    "--prompt", PROMPT, "--category", CATEGORY,
    "--threshold", str(THRESHOLD), "--precision", PRECISION,
    "--output-dir", str(OUTPUT),
], env=RUN_ENV, check=True)
result = json.loads((OUTPUT / "result.json").read_text())
print("Instances:", result["num_instances"])
print("Peak CUDA allocated GiB:", result["peak_cuda_allocated_gib"])
with PILImage.open(OUTPUT / "overlay.png") as preview:
    preview.thumbnail((1200, 1200))
    display(preview.copy())


## 7. 保存到 Drive 并下载结果

输出包括逐实例二值掩码、并集掩码、透明抠图、叠加图和含 score/box/RLE 的 JSON。没有实例通过阈值时，掩码与抠图为空，叠加图保留原图；这不等于证明图中没有目标。

In [ ]:
shutil.copy2(AUDIT_PATH, OUTPUT / "checkpoint-audit.json")
assert (REPO / "SOURCE_MANIFEST.json").is_file(), "缺少源码清单"
shutil.copy2(REPO / "SOURCE_MANIFEST.json", OUTPUT / "source-manifest.json")
result_zip = Path(shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT))
drive_results = Path("/content/drive/MyDrive/SAM3-I/results")
drive_results.mkdir(parents=True, exist_ok=True)
saved_result = drive_results / result_zip.name
shutil.copy2(result_zip, saved_result)
print("Saved:", saved_result)
files.download(str(result_zip))


资源与实现依据：[Colab I/O](https://colab.research.google.com/notebooks/io.ipynb)、[uv Python 管理](https://docs.astral.sh/uv/guides/install-python/)、[uv 锁文件](https://docs.astral.sh/uv/concepts/projects/sync/)、[PyTorch AMP](https://docs.pytorch.org/docs/2.7/amp.html)。